Here is the code to format and train the model on pokedex.csv.  This project trains a Llama 3 8B model using unsloth on Pokemon Data.  First we will download the necessary libraries for the project for Google Colab.  

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install langchain langchain-community sentence-transformers chromadb
!pip install evaluate rouge_score
!pip install -U langchain langchain-community langchain-huggingface chromadb sentence-transformers langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 130.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71

Here is the data formatting section for my project. This specificing formatting is in the ShareGPT style.  I convert the raw CSV file into a structured training dataset.  I load the Pokedex data using pandas, parse the moves column from text to a dictionary, and iterate through each row to build conversational pairs.  Then, I export the final list as a JSONL file.

In [ ]:
# I import the data processing libraries.
# One problem with raw CSV files is they are hard to manipulate directly.
# I fix this by loading the file into a pandas DataFrame.
# This gives me easy access to the rows and columns.
import pandas as pd
import ast
import json

def FormatPokedexDataFull(FileName):
    PokemonData = pd.read_csv(FileName)
    PokemonData.fillna("None", inplace=True)

    # I parse the move strings into dictionaries.
    # One problem with the raw data is the moves are stored as plain text.
    # I fix this by evaluating the literal string structures.
    # This gives me accessible data objects for each move.
    def ParseMoves(MoveString):
        if MoveString == "None":
            return {}
        try:
            return ast.literal_eval(MoveString)
        except (ValueError, SyntaxError):
            return {}

    PokemonData['moves'] = PokemonData['moves'].apply(ParseMoves)

    ShareGptDataset = []
    SystemPrompt = "You are a specialized digital encyclopedia designed to provide highly accurate, analytical, and competitive data regarding Pokémon."

    # I build the ShareGPT conversational dataset.
    # One problem with tabular data is the language model cannot train on it directly.
    # I fix this by iterating through the rows and generating system, user, and assistant messages.
    # This gives me a structured format for fine-tuning.
    for Index, Row in PokemonData.iterrows():

        # Taxonomic facts
        HumanPrompt1 = f"Provide a comprehensive overview of the basic characteristics and taxonomy of {Row['name']}."
        GptResponse1 = f"{Row['name']} ({Row['japanese_name']}) is a Generation {Row['generation']} Pokémon classified as the {Row['species']}. It has a height of {Row['height_m']} meters and a weight of {Row['weight_kg']} kilograms."

        ShareGptDataset.append({
            "conversations": [
                {"role": "system", "content": SystemPrompt},
                {"role": "user", "content": HumanPrompt1},
                {"role": "assistant", "content": GptResponse1}
            ]
        })

        # Competitive analysis
        if Row['smogon_description'] != "None":
            HumanPrompt2 = f"Detail the competitive viability and strategic usage of {Row['name']}."
            GptResponse2 = str(Row['smogon_description'])

            ShareGptDataset.append({
                "conversations": [
                    {"role": "system", "content": SystemPrompt},
                    {"role": "user", "content": HumanPrompt2},
                    {"role": "assistant", "content": GptResponse2}
                ]
            })

        # Combat statistics
        HumanPrompt3 = f"Analyze the statistical distribution of {Row['name']}."
        GptResponse3 = f"{Row['name']} has a Base Stat Total of {Row['total_points']}. The distribution is {Row['hp']} HP, {Row['attack']} Attack, {Row['defense']} Defense, {Row['sp_attack']} Special Attack, {Row['sp_defense']} Special Defense, and {Row['speed']} Speed."

        ShareGptDataset.append({
            "conversations": [
                {"role": "system", "content": SystemPrompt},
                {"role": "user", "content": HumanPrompt3},
                {"role": "assistant", "content": GptResponse3}
            ]
        })

        # Elemental multipliers
        MatchupText = f"{Row['name']} takes the following damage from attacks: "
        MatchupList = []
        TypesList = ['normal', 'fire', 'water', 'electric', 'grass', 'ice', 'fighting', 'poison', 'ground', 'flying', 'psychic', 'bug', 'rock', 'ghost', 'dragon', 'dark', 'steel', 'fairy']

        for TypeName in TypesList:
            ColumnName = f"against_{TypeName}"
            if ColumnName in Row:
                MultiplierValue = Row[ColumnName]
                if MultiplierValue == 0.0:
                    MatchupList.append(f"an immunity to {TypeName}")
                elif MultiplierValue == 0.5 or MultiplierValue == 0.25:
                    MatchupList.append(f"a resistance to {TypeName}")
                elif MultiplierValue == 2.0 or MultiplierValue == 4.0:
                    MatchupList.append(f"a weakness to {TypeName}")

        if MatchupList:
            MatchupText += ", ".join(MatchupList) + "."
            HumanPrompt4 = f"What are the elemental matchups and weaknesses for {Row['name']}?"
            ShareGptDataset.append({
                "conversations": [
                    {"role": "system", "content": SystemPrompt},
                    {"role": "user", "content": HumanPrompt4},
                    {"role": "assistant", "content": MatchupText}
                ]
            })

        # Attack mechanics
        for MoveName, MoveData in Row['moves'].items():
            if isinstance(MoveData, dict):
                HumanPrompt5 = f"How does the move {MoveName} function when utilized by {Row['name']}?"
                MoveType = MoveData.get('Type', 'Unknown')
                MovePower = MoveData.get('Power', '0')
                MoveAccuracy = MoveData.get('Accuracy', '0')
                MoveDesc = MoveData.get('Description', '')
                GptResponse5 = f"{MoveName} is a {MoveType}-type attack with {MovePower} Base Power and {MoveAccuracy}% accuracy. {MoveDesc}"

                ShareGptDataset.append({
                    "conversations": [
                        {"role": "system", "content": SystemPrompt},
                        {"role": "user", "content": HumanPrompt5},
                        {"role": "assistant", "content": GptResponse5}
                    ]
                })

    # I export the dataset to a file.
    # One problem with memory objects is they do not persist.
    # I fix this by writing the data to a JSONL file.
    # This gives me a saved dataset ready for the training script.
    with open("PokedexShareGptFull.jsonl", "w") as OutFile:
        for Item in ShareGptDataset:
            json.dump(Item, OutFile)
            OutFile.write('\n')

FormatPokedexDataFull('pokedex.csv')

/tmp/ipykernel_2768/2860340532.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'None' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  PokemonData.fillna("None", inplace=True)


I prepare the base language model for fine-tuning. You can see my memory management setup below. I apply 4-bit quantization and use bfloat16 to reduce hardware demands. I attach Low-Rank Adaptation matrices to the model to target specific modules. This configuration lets me train the model efficiently.

In [ ]:
from unsloth import FastLanguageModel
import torch

MaxSeqLength = 2048
DType = torch.bfloat16
LoadIn4Bit = True

# I initialize the base language model.
# One problem with large models is they consume too much video memory.
# I fix this by loading a 4-bit quantized version of the Llama model.
# This gives me a model that fits on standard graphics hardware.
Model, Tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = MaxSeqLength,
    dtype = DType,
    load_in_4bit = LoadIn4Bit,
)

# I configure the model for efficient fine-tuning.
# One problem with standard training is updating all parameters takes too much time.
# I fix this by applying Low-Rank Adaptation matrices to specific target modules.
# This gives me a fast training phase while maintaining model quality.
Model = FastLanguageModel.get_peft_model(
    Model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


I prepare the training script for my Pokemon LLM. You can review my exact configuration below. I load the custom data and format it into a standard chat structure. I set the trainer to ignore the user questions. This forces the model to learn only from the assistant responses.

In [ ]:
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported, unsloth_train
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, train_on_responses_only

MaxSeqLength = 2048

# I load the base language model.
# One problem with large models is they require heavy computing power.
# I fix this by loading a smaller 4-bit version.
# This gives me a model I can train on my local machine.
Model, Tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = MaxSeqLength,
    dtype = torch.bfloat16,
    load_in_4bit = True,
)

# I configure the model for targeted training.
# One problem with standard training is it updates every single parameter.
# I fix this by applying a targeted adaptation method to specific modules.
# This gives me a fast training process.
Model = FastLanguageModel.get_peft_model(
    Model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Tokenizer = get_chat_template(
    Tokenizer,
    chat_template = "llama-3.1"
)

# I prepare the custom training data.
# One problem with raw text files is the training script cannot read them.
# I fix this by loading the JSON file and applying the standard chat template.
# This gives me properly formatted inputs for the trainer.
Dataset = load_dataset("json", data_files="PokedexShareGptFull.jsonl", split="train")
Dataset = standardize_sharegpt(Dataset)

def FormatChatTemplate(Examples):
    Texts = [Tokenizer.apply_chat_template(Message, tokenize=False, add_generation_prompt=False) for Message in Examples["conversations"]]
    return {"text": Texts}

Dataset = Dataset.map(FormatChatTemplate, batched=True)

# I define the training parameters.
# One problem with default training settings is they do not fit my specific hardware or data size.
# I fix this by setting a custom batch size, learning rate, and step count.
# This gives me an optimized training loop.
Trainer = SFTTrainer(
    model = Model,
    tokenizer = Tokenizer,
    train_dataset = Dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MaxSeqLength,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        fp16 = False,
        bf16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# I mask the user prompts during training.
# One problem with standard chat training is the model learns to copy the user questions.
# I fix this by telling the trainer to only learn from the assistant responses.
# This gives me a model that focuses entirely on answering accurately.
Trainer = train_on_responses_only(
    Trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

TrainerStats = unsloth_train(Trainer)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Standardizing formats (num_proc=52):   0%|          | 0/17609 [00:00<?, ? examples/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=52):   0%|          | 0/17609 [00:00<?, ? examples/s]

Filter (num_proc=52):   0%|          | 0/17609 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17,609 | Num Epochs = 2 | Total steps = 2,202
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
1,2.251057
2,2.010101
3,2.254069
4,2.304831
5,2.337370
6,2.041260
7,2.296156
8,2.207597
9,2.032612
10,1.965171


Now we save the models.

In [ ]:
# Save the model and adapters for use in Streamlit/RAG
Model.save_pretrained("lora_model")
Tokenizer.save_pretrained("lora_model")

print("Part 2 Complete")

Phase 2 Complete: Model fine-tuned and adapters saved to 'lora_model' directory.


Here, I setup the vector database. I process the raw data and embed it into a local storage file. This builds the retrieval system for the language model.

In [ ]:
import pandas as PandasLibrary
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

ParsedDocuments = []

# I load the source data into a table format.
# One problem with unstructured files is the search retriever cannot read them easily.
# I fix this by parsing the CSV file using the pandas library.
# This gives me an organized data structure for the vector store.
DataFrame = PandasLibrary.read_csv("pokedex.csv")
DataFrame.fillna("None", inplace=True)

# I create search documents for the vector database.
# One problem with isolated facts is the retriever loses context.
# I fix this by combining the Pokémon name and description into a single string.
# This gives me highly relevant search vectors.
for IndexValue, RowData in DataFrame.iterrows():
    CombinedContent = f"Pokemon: {RowData['name']}\nInformation: {RowData['bulba_description']}"
    NewDocument = Document(page_content=CombinedContent)
    ParsedDocuments.append(NewDocument)

# I set up the embedding model.
# One problem with raw text is the database cannot calculate similarity.
# I fix this by loading a pre-trained HuggingFace model.
# This gives me numerical representations of the text.
ModelName = "BAAI/bge-large-en-v1.5"
ModelKwargs = {'device': 'cuda'}
EncodeKwargs = {'normalize_embeddings': True}

EmbeddingsModel = HuggingFaceEmbeddings(
    model_name=ModelName,
    model_kwargs=ModelKwargs,
    encode_kwargs=EncodeKwargs
)

# I build the local vector database.
# One problem with memory-based indexes is they delete when the script ends.
# I fix this by setting a persistent directory for Chroma.
# This gives me a saved index I can use later.
PersistDirectory = "chroma_db"
VectorDatabase = Chroma.from_documents(
    documents=ParsedDocuments,
    embedding=EmbeddingsModel,
    persist_directory=PersistDirectory
)

print(f"Part 3 Complete")

/tmp/ipykernel_2768/74018652.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'None' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  DataFrame.fillna("None", inplace=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Phase 3 Indexing Complete: 1045 chunks stored in 'chroma_db'


Now we save the model.

In [ ]:
Model.save_pretrained_merged(
    "pokedex_llama3_model",
    Tokenizer,
    save_method = "merged_16bit"
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:51<00:00, 12.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:23<00:00, 20.80s/it]


Unsloth: Merge process complete. Saved to `/content/pokedex_llama3_model`


I configure the pipeline setup for my model inference. I load the 16-bit language model into memory. I then set the text generation parameters to stop run-on sentences. Finally, I wrap the generator so I can plug it directly into my local vector database.

In [ ]:
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from unsloth import FastLanguageModel

# I load the fine-tuned 16-bit model.
# One problem with quantized weights is they lose accuracy during inference.
# I fix this by loading the model in full bfloat16 precision.
# This gives me the best possible text outputs.
Model, Tokenizer = FastLanguageModel.from_pretrained(
    model_name = "pokedex_llama3_model",
    max_seq_length = 2048,
    dtype = torch.bfloat16,
    load_in_4bit = False,
)

# I prepare the model for text generation.
# One problem with loaded models is they retain training overhead.
# I fix this by calling the native Unsloth inference function.
# This gives me a highly optimized generation speed.
FastLanguageModel.for_inference(Model)

# I define the termination tokens.
# One problem with the Llama 3.1 architecture is it ignores standard stop commands.
# I fix this by converting the specific end-of-turn string to an ID.
# This gives me clean answers that stop exactly when the assistant finishes talking.
MyTerminators = [
    Tokenizer.eos_token_id,
    Tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

# I create the local generation pipeline.
# One problem with default generation is the text wanders off topic.
# I fix this by setting strict sampling parameters and repetition penalties.
# This gives me highly focused and relevant information blocks.
Generator = pipeline(
    "text-generation",
    model = Model,
    tokenizer = Tokenizer,
    max_new_tokens = 512,
    temperature = 0.4,
    top_p = 0.9,
    min_p = 0.05,
    repetition_penalty = 1.1,
    do_sample = True,
    eos_token_id = MyTerminators,
    pad_token_id = Tokenizer.eos_token_id,
)

# I initialize the LangChain wrapper.
# One problem with the raw generator is it cannot connect to external databases.
# I fix this by passing the pipeline into HuggingFacePipeline.
# This gives me an object I can plug directly into the retrieval system.
LocalLlm = HuggingFacePipeline(pipeline = Generator)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer you are loading from 'pokedex_llama3_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'pokedex_llama3_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load pokedex_llama3_model as a legacy tokenizer.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty', 'top_p', 'eos_token_id', 'pad_token_id', 'min_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass e

Now I save the files to run on my local computer as a streamlit app.

In [ ]:
import os
from google.colab import files

!zip -r pokemon_assistant_project.zip chroma_db pokedex.csv

files.download("pokemon_assistant_project.zip")

	zip warning: name not matched: final_dataset_with_abilities.csv
  adding: lora_model/ (stored 0%)
  adding: lora_model/README.md (deflated 65%)
  adding: lora_model/tokenizer.json (deflated 85%)
  adding: lora_model/adapter_config.json (deflated 57%)
  adding: lora_model/tokenizer_config.json (deflated 45%)
  adding: lora_model/chat_template.jinja (deflated 72%)
  adding: lora_model/adapter_model.safetensors (deflated 7%)
  adding: chroma_db/ (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 36%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/ (stored 0%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/header.bin (deflated 58%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/data_level0.bin (deflated 9%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/length.bin (deflated 80%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/link_lists.bin (deflated 85%)
  adding: chroma_db/6f863f32-23e2-42b7-bc63-ee4a902b0109/index_metadata.pickle (

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This specific model is too large, it needs to be saved on google drive.

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/Drive')

SourcePath = "pokedex_llama3_model.zip"
DestinationPath = "/content/Drive/MyDrive/pokedex_llama3_model.zip"

shutil.copy(SourcePath, DestinationPath)

Mounted at /content/Drive


'/content/Drive/MyDrive/pokedex_llama3_model.zip'